# Collaborative Filtering with K-Nearest Neighbors

This notebook implements the KNN collaborative-filtering model described in the presentation. KNN predicts a user's rating for a course from similar users or similar courses.

## Dependency note

The original capstone uses `scikit-surprise` for KNN collaborative filtering. Install the requirements before running this notebook.

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

DATA_DIR = Path("datasets")
DATA_URLS = {
    "ratings.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv",
    "course_genre.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_genre.csv",
    "rs_content_test.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/rs_content_test.csv",
    "user_profile.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/user_profile.csv",
    "course_processed.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv",
    "courses_bows.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/courses_bows.csv",
    "sim.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/sim.csv",
}

def ensure_dataset(filename):
    DATA_DIR.mkdir(exist_ok=True)
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(DATA_URLS[filename], path)
    return path

def load_csv(filename, **kwargs):
    return pd.read_csv(ensure_dataset(filename), **kwargs)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

In [ ]:
from surprise import Dataset, Reader, KNNBasic, accuracy
from surprise.model_selection import train_test_split

ratings_df = load_csv("ratings.csv")
ratings_df.head()

In [ ]:
reader = Reader(rating_scale=(2, 3))
data = Dataset.load_from_df(ratings_df[["user", "item", "rating"]], reader)
trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

sim_options = {
    "name": "cosine",
    "user_based": False,
}

knn_model = KNNBasic(k=100, min_k=5, sim_options=sim_options, verbose=True)
knn_model.fit(trainset)
knn_predictions = knn_model.test(testset)
knn_rmse = accuracy.rmse(knn_predictions)

print(f"KNN RMSE: {knn_rmse:.4f}")

## Interpretation

KNN is simple and explainable: it recommends based on neighboring users/items. The presentation reports RMSE around `0.2063`. KNN is fast to prototype but can struggle as the user-item matrix becomes sparse and large.